In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *


In [2]:
spark = SparkSession.builder \
    .appName("AmazonReviews") \
    .config("spark.driver.memory", "24g") \
    .config("spark.executor.memory", "24g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.default.parallelism", "200") \
    .config("spark.driver.maxResultSize", "4g") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/29 07:37:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
df = spark.read.parquet(
    '/kaggle/input/datasets/datdong123/amazon-clean-v2/amazon_reviews_cleaned_v3'
)

In [4]:
df.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- review_id: string (nullable = true)
 |-- product_parent: string (nullable = true)
 |-- product_title: string (nullable = true)
 |-- star_rating: integer (nullable = true)
 |-- helpful_votes: integer (nullable = true)
 |-- total_votes: integer (nullable = true)
 |-- vine: integer (nullable = true)
 |-- verified_purchase: integer (nullable = true)
 |-- review_headline: string (nullable = true)
 |-- review_body: string (nullable = true)
 |-- review_date: date (nullable = true)
 |-- helpful_ratio: double (nullable = true)
 |-- review_length: integer (nullable = true)
 |-- product_category: string (nullable = true)



TRAIN TEST SPLIT (TIME BASED)

In [5]:
from pyspark.sql.functions import col

df_ts = df.withColumn(
    "review_ts", col("review_date").cast("timestamp").cast("long")
)

In [6]:
quantiles = df_ts.approxQuantile("review_ts", [0.8, 0.9], 0.01)

train_cutoff = quantiles[0]
val_cutoff = quantiles[1]

In [7]:
train_df = df_ts.filter(col("review_ts") <= train_cutoff)

val_df = df_ts.filter(
    (col("review_ts") > train_cutoff) &
    (col("review_ts") <= val_cutoff)
)

test_df = df_ts.filter(col("review_ts") > val_cutoff)

# -----------------------------
# 1. INDEXING (fit ONLY on train)
# -----------------------------

In [8]:
from pyspark.ml.feature import StringIndexer

user_indexer = StringIndexer(
    inputCol="customer_id",
    outputCol="user_id",
    handleInvalid="skip"
)

item_indexer = StringIndexer(
    inputCol="product_id",
    outputCol="item_id",
    handleInvalid="skip"
)

# Fit ONLY on train
user_model = user_indexer.fit(train_df)
item_model = item_indexer.fit(train_df)

train_df = user_model.transform(train_df)
train_df = item_model.transform(train_df)

val_df = user_model.transform(val_df)
val_df = item_model.transform(val_df)

test_df = user_model.transform(test_df)
test_df = item_model.transform(test_df)

# -----------------------------
# 2. REPARTITION + CACHE
# -----------------------------

In [9]:
train_df = train_df.repartition(200, "user_id")
train_df.cache()
train_df.count()

26/03/29 07:38:34 WARN DAGScheduler: Broadcasting large task binary with size 148.5 MiB
26/03/29 07:44:55 WARN DAGScheduler: Broadcasting large task binary with size 42.2 MiB
26/03/29 07:48:22 WARN DAGScheduler: Broadcasting large task binary with size 42.2 MiB


25379405

# -----------------------------
# 3. EVALUATORS
# -----------------------------

In [10]:
from pyspark.ml.feature import StringIndexer
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator

evaluator_rmse = RegressionEvaluator(
    metricName="rmse",
    labelCol="star_rating",
    predictionCol="prediction"
)

evaluator_mae = RegressionEvaluator(
    metricName="mae",
    labelCol="star_rating",
    predictionCol="prediction"
)

# -----------------------------
# 4. MANUAL HYPERPARAM TUNING 
# -----------------------------

In [11]:
from pyspark.storagelevel import StorageLevel

# Cache validation (important)
val_df.persist(StorageLevel.MEMORY_AND_DISK)
val_df.count()


ranks = [10, 20]          
regParams = [0.01, 0.1]
maxIters = [5]           

best_rmse = float("inf")
best_model = None
best_params = None

for rank in ranks:
    for reg in regParams:
        for maxIter in maxIters:

            print(f"\nTraining: rank={rank}, reg={reg}, maxIter={maxIter}")

            als = ALS(
                userCol="user_id",
                itemCol="item_id",
                ratingCol="star_rating",
                coldStartStrategy="drop",
                nonnegative=True,
                rank=rank,
                regParam=reg,
                maxIter=maxIter,
                seed=42,

                # 🔥 CRITICAL FOR MEMORY
                numUserBlocks=10,
                numItemBlocks=10
            )

            try:
                model = als.fit(train_df)

                preds = model.transform(val_df)

                rmse = evaluator_rmse.evaluate(preds)

                print(f"RMSE={rmse}")

                if rmse < best_rmse:
                    best_rmse = rmse
                    best_model = model
                    best_params = (rank, reg, maxIter)

            except Exception as e:
                print("❌ Failed due to:", e)

print("\nBest params:", best_params)
print("Best validation RMSE:", best_rmse)

26/03/29 07:50:14 WARN DAGScheduler: Broadcasting large task binary with size 148.5 MiB
26/03/29 07:56:08 WARN DAGScheduler: Broadcasting large task binary with size 148.5 MiB



Training: rank=10, reg=0.01, maxIter=5


26/03/29 08:00:44 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/03/29 08:00:51 WARN DAGScheduler: Broadcasting large task binary with size 42.3 MiB
26/03/29 08:00:59 WARN DAGScheduler: Broadcasting large task binary with size 42.3 MiB
26/03/29 08:02:49 WARN DAGScheduler: Broadcasting large task binary with size 42.3 MiB
26/03/29 08:04:39 WARN DAGScheduler: Broadcasting large task binary with size 42.3 MiB
26/03/29 08:04:59 WARN DAGScheduler: Broadcasting large task binary with size 42.3 MiB
26/03/29 08:06:50 WARN DAGScheduler: Broadcasting large task binary with size 42.3 MiB
26/03/29 08:07:04 WARN DAGScheduler: Broadcasting large task binary with size 42.3 MiB
26/03/29 08:07:15 WARN DAGScheduler: Broadcasting large task binary with size 42.3 MiB
26/03/29 08:07:36 WARN DAGScheduler: Broadcasting large task binary with size 42.3 MiB
26/03/29 08:08:29 WARN DAGSc

RMSE=2.3433011889019646

Training: rank=10, reg=0.1, maxIter=5


26/03/29 08:21:41 WARN DAGScheduler: Broadcasting large task binary with size 42.3 MiB
26/03/29 08:21:48 WARN DAGScheduler: Broadcasting large task binary with size 42.3 MiB
26/03/29 08:23:37 WARN DAGScheduler: Broadcasting large task binary with size 42.3 MiB
26/03/29 08:25:27 WARN DAGScheduler: Broadcasting large task binary with size 42.3 MiB
26/03/29 08:25:44 WARN DAGScheduler: Broadcasting large task binary with size 42.3 MiB
26/03/29 08:27:31 WARN DAGScheduler: Broadcasting large task binary with size 42.3 MiB
26/03/29 08:27:46 WARN DAGScheduler: Broadcasting large task binary with size 42.3 MiB
26/03/29 08:27:55 WARN DAGScheduler: Broadcasting large task binary with size 42.3 MiB
26/03/29 08:28:13 WARN DAGScheduler: Broadcasting large task binary with size 42.3 MiB
26/03/29 08:28:50 WARN DAGScheduler: Broadcasting large task binary with size 42.3 MiB
26/03/29 08:29:10 WARN DAGScheduler: Broadcasting large task binary with size 42.3 MiB
26/03/29 08:29:44 WARN DAGScheduler: Broadc

RMSE=1.4942316178533102

Training: rank=20, reg=0.01, maxIter=5


26/03/29 08:40:45 WARN DAGScheduler: Broadcasting large task binary with size 42.3 MiB
26/03/29 08:40:53 WARN DAGScheduler: Broadcasting large task binary with size 42.3 MiB
26/03/29 08:42:48 WARN DAGScheduler: Broadcasting large task binary with size 42.3 MiB
26/03/29 08:44:41 WARN DAGScheduler: Broadcasting large task binary with size 42.3 MiB
26/03/29 08:44:59 WARN DAGScheduler: Broadcasting large task binary with size 42.3 MiB
26/03/29 08:46:49 WARN DAGScheduler: Broadcasting large task binary with size 42.3 MiB
26/03/29 08:47:04 WARN DAGScheduler: Broadcasting large task binary with size 42.3 MiB
26/03/29 08:47:17 WARN DAGScheduler: Broadcasting large task binary with size 42.3 MiB
26/03/29 08:47:42 WARN DAGScheduler: Broadcasting large task binary with size 42.3 MiB
26/03/29 08:49:27 WARN DAGScheduler: Broadcasting large task binary with size 42.3 MiB
26/03/29 08:50:07 WARN DAGScheduler: Broadcasting large task binary with size 42.3 MiB
26/03/29 08:51:38 WARN DAGScheduler: Broadc

RMSE=2.23594496414585

Training: rank=20, reg=0.1, maxIter=5


26/03/29 09:08:40 WARN DAGScheduler: Broadcasting large task binary with size 42.3 MiB
26/03/29 09:08:47 WARN DAGScheduler: Broadcasting large task binary with size 42.3 MiB
26/03/29 09:10:41 WARN DAGScheduler: Broadcasting large task binary with size 42.3 MiB
26/03/29 09:12:34 WARN DAGScheduler: Broadcasting large task binary with size 42.3 MiB
26/03/29 09:12:53 WARN DAGScheduler: Broadcasting large task binary with size 42.3 MiB
26/03/29 09:14:43 WARN DAGScheduler: Broadcasting large task binary with size 42.3 MiB
26/03/29 09:14:56 WARN DAGScheduler: Broadcasting large task binary with size 42.3 MiB
26/03/29 09:15:10 WARN DAGScheduler: Broadcasting large task binary with size 42.3 MiB
26/03/29 09:15:31 WARN DAGScheduler: Broadcasting large task binary with size 42.3 MiB
26/03/29 09:16:27 WARN DAGScheduler: Broadcasting large task binary with size 42.3 MiB
26/03/29 09:16:57 WARN DAGScheduler: Broadcasting large task binary with size 42.3 MiB
26/03/29 09:17:40 WARN DAGScheduler: Broadc

RMSE=1.4931590963159296

Best params: (20, 0.1, 5)
Best validation RMSE: 1.4931590963159296


# -----------------------------
# 5. FINAL TEST EVALUATION
# -----------------------------

In [12]:
test_preds = best_model.transform(test_df)

test_rmse = evaluator_rmse.evaluate(test_preds)
test_mae = evaluator_mae.evaluate(test_preds)

print("\nTest RMSE:", test_rmse)
print("Test MAE:", test_mae)

26/03/29 09:32:09 WARN DAGScheduler: Broadcasting large task binary with size 42.3 MiB
26/03/29 09:32:16 WARN DAGScheduler: Broadcasting large task binary with size 42.3 MiB
26/03/29 09:32:26 WARN DAGScheduler: Broadcasting large task binary with size 148.5 MiB
26/03/29 09:37:35 WARN DAGScheduler: Broadcasting large task binary with size 148.6 MiB
26/03/29 09:43:41 WARN DAGScheduler: Broadcasting large task binary with size 148.6 MiB
26/03/29 09:49:34 WARN DAGScheduler: Broadcasting large task binary with size 148.6 MiB
26/03/29 09:50:01 WARN DAGScheduler: Broadcasting large task binary with size 42.3 MiB
26/03/29 09:50:07 WARN DAGScheduler: Broadcasting large task binary with size 42.3 MiB
26/03/29 09:50:19 WARN DAGScheduler: Broadcasting large task binary with size 148.5 MiB
26/03/29 09:55:28 WARN DAGScheduler: Broadcasting large task binary with size 148.6 MiB
26/03/29 10:01:53 WARN DAGScheduler: Broadcasting large task binary with size 148.6 MiB
26/03/29 10:07:58 WARN DAGScheduler:


Test RMSE: 1.5247493032093784
Test MAE: 1.2097255787517271


In [13]:
base_path = "/kaggle/working/als_model"

als_path = f"{base_path}/als"
user_indexer_path = f"{base_path}/user_indexer"
item_indexer_path = f"{base_path}/item_indexer"

# Save ALS model
best_model.write().overwrite().save(als_path)

# Save indexers (VERY IMPORTANT)
user_model.write().overwrite().save(user_indexer_path)
item_model.write().overwrite().save(item_indexer_path)

print("Model saved to:", base_path)

26/03/29 10:08:23 WARN DAGScheduler: Broadcasting large task binary with size 42.5 MiB
26/03/29 10:08:37 WARN DAGScheduler: Broadcasting large task binary with size 42.5 MiB
26/03/29 10:08:43 WARN TaskSetManager: Stage 538 contains a task of very large size (50322 KiB). The maximum recommended task size is 1000 KiB.
26/03/29 10:08:45 WARN TaskSetManager: Stage 540 contains a task of very large size (17450 KiB). The maximum recommended task size is 1000 KiB.


Model saved to: /kaggle/working/als_model
